# ▂▂▂▂▂▂▂▂▂▂▂▂

# Write-Up

## LinkedSpace Model

I created a new model variant which isn't too dissimilar from the existing `SharedSpaceDecoder` model, but is designed to provide more freedom in how we configure the spaces.

Currently it has two main new features:

1. It supports arbitrary linking / tying of the shared spaces.
2. It allows for specifying arbitrary patterns of shared space layers and dense layers.



### Tying Shared Spaces

**Linked Spaces**

In the SharedSpaceDecoder, the Key-Value subspace is unique in that, in addition to being shared by all heads in a layer (this is what we refer to as a "shared space"), it is also 'shared' by both the Key and the Value heads.

To provide some alternative language to distinguish between heads sharing a space vs. modules sharing a space, I'm currently using the term "linked spaces". So the Key and Value modules have their shared spaces "linked".

Another common term would be "weight tying".(Update: I'm actually going with 'tied' instead of 'linked', but 'linked' is what's used in this notebook 😊).



**Ideas for Linking**

I've been wanting to explore various linkings.

For example:
* What happens if we link the Q and O spaces?
* Or Q and K?

I've also wanted to extend this to shared spaces on the FFNs.

> Note: In attention, a "shared space" is shared across heads; in the context of an FFN, a "shared space" is shared across neurons.

For the FFN:
* What if we link the attention O space to the input spaces ($W^{in}$ and $W^{gate}$) of the FFN?


**Configuration**

To allow us to explore arbitrary space linkings, I added a `spaces` object to the model configuration which can look like the below example:

```python
"spaces": {
    "0": {
      "size": -1,
      "modules": ["Q", "O", "out"]
    },
    "kv": {
      "modules": ["K", "V"],
      "size": 128
    },
    "ingt": {
      "size": 192,
      "modules": ["in", "gate"]
    }
}
```

## Equal Parameter Counts

In our last round of experiments, an important observation was that any shared subspace--whether for the queries, the output, or the keys and values:
1. Reduces parameter count,
2. Improves speed,
3. Reduces accuracy.

The amount of speed improvement and amount of accuracy lost definitely correlates, at a glance, with the parameter count.

This caused me to reframe our research question as “what is the best allocation of parameters?” e.g., is it better to put a shared subspace on the queries, or on the output, or both?

### Normalizing Accuracy for Parameters



The first direction I explored for this was to simply divide accuracy by parameter count.

This rewards our SharedSpace models for having fewer parameters / punishes the dense baseline for having too many. With this approach, suddenly our models look far better than the baseline.

However, I think this linear relationship is too generous to the decomposed models. My suspicion comes from the fact that, whenever I've tried increasing the size of one of our "sparse" models, hoping to outperform the dense baseline, it hasn't worked.

I asked GPT about how people relate parameter count to performance, and it said that there's generally thought to be a more exponential relationship between the two--i.e., a linear improvement in accuracy requires exponentially more parameters.

Using the equations it suggested, the baseline still ranked first as having the best accuracy per parameter.

TODO - Share the math, and maybe some examples of under-performance.


### Requiring Equal Parameters

Since the normalization approach can only be an estimate of how the models compare, I decided to try adding a constraint to our benchmarks--the models must all use the same number of parameters as the dense baseline.

I'm liking this approach, because it makes sense to me that, if our models are truly "efficient", then they ought to outperform the dense model when we increase their size to match.

### Depth vs. Width

I looked at an older paper from Google: _Scale Efficiently: Insights from Pre-training and Fine-tuning Transformers_ about how best to allocate additional parameters in a model.

They found that it's generally best to increase the depth of the model rather than its "width" (the size of a layer).

I think this makes some intuitive sense--there's only so much the model can accomplish with the current state of the residual stream before it needs to aggregate the results in order to progress.

Based on that insight plus my gut intuition from "playing around" with different configurations, it seems that, for a given model dimension (embedding size), there's roughly an ideal layer configuration (number of heads and number of neurons).

This aligns with the established conventions:
* `head_size = d_model / num_heads`
* `num_neurons = d_model * 4`
    * Or `num_neurons = d_model * 4 * 2/3` in the case of SwiGLU.

>Note: Those conventions don't fully tell you what the head count / size should be. I typically follow:
> - For `d_model = 256, d_head = 32`
> - For `d_model = 512 or 768, d_head = 64`
>
> I haven't trained larger than that, but it does seem (from looking at model configurations) that 128 is a good upper-bound on head size.
  


### Configuring 'Identical' Parameter Counts

The biggest challenge with this "equal parameters" approach is that it seems to be impossible to make the parameter counts identical, and just getting them to be "very close" requires choosing unusual dimensions.

Here's the approach I took in my experiments:

* I kept `d_model` constant, at 256.
* I kept `d_head` constant, at 32.

To decrease the parameter count (i.e., to find a more "effecient" architecture), I played with:

* Subspace configuration (i.e., which modules had a shared space, and which shared spaces were linked)
* Subspace sizes
* Percentage change in both number of heads and number of neurons.
    * For example:
        * 8 heads --> 672 neurons
        * 6 heads --> 512 neurons  (6/8)
        * 3 heads --> 256 neurons  (3/8)

I generally tried to work out configurations that would allow for the highest number of layers without decreasing their width "too much".

Once the layer count was set, I would manually dial in the total parameter count to match the baseline as closely as possible by adjusting the number of heads and the number of neurons.

Typically, I could find a way to match the parameter count to the dense baseline within 20,000 parameters (about 0.2%)


### Results

**Dense Baseline**

The dense baseline consistently had the lowest perplexity.

It's configuration is:
* `d_model = 256`
* `n_layers = 6`
* `n_heads = 8`
* `d_head = 32`
* `n_neurons = 672`
    * This is based on `256 * 4 * 2/3 = ~683` for SwiGLU
    * `672` is `21 * 32`.

It has 17,546,368 parameters (or 16.73M in base-2).



**Sparse Models**

I had better results when I mixed in dense layers, discussed in the next section.

Here are the results with only sparse layers.

It's interesting to see how relatively consistent the Transformer is able to perform across all of these wildly different arrangements.

The full training run was for 3.3K steps, but I typically stopped a run early when I felt like I had enough of a trend and was ready to move on to the next experiment.

It's difficult to read the data here, but I'll share a few notes.


<img src='https://lh3.googleusercontent.com/d/1dWnrMHBQEf4_k2W77do3__VHewoMNMUX' alt='Screenshot' width='900' />

Some observations:

* An extreme configuration of 14 layers, 4 heads, and a small MLP with 344 neurons, performed very poorly--that's the top orange line.


The below three runs look like they might be worth exploring more. They seem to suggest that 8 sparser layers (yellow and mint) are better than 7 larger / denser ones (light blue), and it'd be great to "prove" this more.

<img src='https://lh3.googleusercontent.com/d/1nVg45RCf6jDGjqluWkv_LebBEy5zmhAZ' alt='Screenshot' width='900' />

The yellow run starts to suggest a kind of ideal ratio for the sparser layers, perhaps

* I eventually focused my attention on applying a linked shared space to the key and value heads, and another to the input and gate neurons.


## Mixing Shared Space Layers and "Dense" Layers

I struggled to find any "shared space" configuration which outperformed the dense baseline on eval perplexity or training loss.

This lead me to theorize that these shared spaces are fundamentally limiting to the model's capabilities--that the model really needs, at least in some layers, full read and write access to the residual stream.



### Sparse-Dense Pattern

I added another configuration option to our model which allows us to specify an arbitrary arrangement of shared space ("sparse") and dense layers.

I further hypothesized that, because it has limited access to the residual stream, an optimal sparse layer probably has fewer heads and fewer neurons than a dense one. I updated the configuration to allow for specifying these values separately for the dense layers vs. the sparse ones.



```python
    "dense_attn_heads": 8,
    "dense_intrmd_size": 672,
  
    "interleave_dense": "dssdssdssd",

    "shrd_attn_heads": 3,
    "shrd_intrmd_size": 256,
    
    "spaces": {
        "0": {
        "size": -1,
        "modules": ["Q", "O", "out"]
        },
        "kv": {
        "modules": ["K", "V"],
        "size": 128
        },
        "ingt": {
        "size": 192,
        "modules": ["in", "gate"]
        }
    }  
```


<img src='https://lh3.googleusercontent.com/d/1Jhvm_Spimb_kVpXgjAfkooTp3o_IQ-1-' alt='Screenshot' width='900' />

Zooming in...

<img src='https://lh3.googleusercontent.com/d/1-CtnVNo_VKe1Nh-MVtLlA-U8Ml32P7Az' alt='Screenshot' width='900' />

### Finally Outperforming the Baseline!

I played with various orderings, and--at long last--arrived at a configuration which achieved a better loss curve and lower perplexity than the dense baseline.



<img src='https://lh3.googleusercontent.com/d/1x0mZD3A8lgFIfoGrZlvE6sQEfGundZts' alt='Screenshot' width='900' />

Results are all [here](https://wandb.ai/chrismccormick/linkdecoder-pretrain-wiki/) on wandb.

**GPT-2 small scale**

The above plots are all from runs at `tiny` scale (6 layers, d_model = 256). I did a handful of runs applying some of the same insights at small scale (12 layers, d_model = 768).

The deep models (e.g., 26 layers, or 31 layers) have a better loss curve initially, but then encounter some bumps which cause them to fall behind.

For example, the grey line below is a 31 layer variant which is the best performing until it has a bump in its loss around 420 steps.

<img src='https://lh3.googleusercontent.com/d/1v5obZNIO4AKjiZITd_hbXB36wOO3L3eU' alt='Screenshot' width='900' />

# Take-Aways & Next Steps

The above experiments, where we attempt to hold the parameter count constant, seem very difficult to change into a structured experiment (at least, I'm not sure yet how we'd do that).

The MASA paper inspired me to instead choose hyperparameters such that all variants have the same number of parameters, yet still less than the baseline.

Their paper explores all of the options at a 2/3s compression rate of the total attention weights.

We're still working out the experiment design for this.

**Narrow Layers vs. Subspace Groups**

I realized that the technique I was applying of reducing the width of the layers while increasing the number is nearly the same as the concept of subspace groups.

For example, if you divide a layer with 12 heads into two layers with 6 heads, then each layer will have their own shared subspace, and you've essentially formed two groups of six heads each, where each group has its own shared subspace.

The difference is that here, rather than running the groups in parallel in a single layer, we are executing them sequentially.

I'm thinking to try the subspace grouping next. Dividing into groups feels like a more structured way to approach this than breaking them into layers. (Though it's possible that the additional layers were what made it work!)

# ▂▂▂▂▂▂▂▂▂▂▂▂

# Experiments

To allow for quick iterations on the code, this time around I tried just importing everything into the Colab Notebook using a utility of mine which helps with that (i.e., it breaks up scripts and creates cells with headings).

# Setup

In [ ]:
# Set a flag we can use to determine if we are running within a Colab instance.
is_colab = "google.colab" in str(get_ipython())

## FlashAttention on Colab

FlashAttention does not come pre-installed on Colab instances, and is very time consuming to install manually because it has to be built from source.

The below GitHub repo, however, provides pre-built wheels which make setup easy.

https://github.com/mjun0812/flash-attention-prebuild-wheels/releases

The key is just to identify the correct wheel to use from the giant list.

We need the wheel specific to our version of python, pytorch, and CUDA. So first we'll print those out:

In [ ]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("")
print("GPU:", torch.cuda.get_device_name(0))

Python: 3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]
PyTorch: 2.8.0+cu126
CUDA: 12.6

GPU: NVIDIA A100-SXM4-80GB


It's difficult to find the correct wheel because they are all hidden underneath different releases, and searching the page doesn't work unless the releases are expanded.

With some hunting, I was able to find the correct version for Colab's current configuration:

In [ ]:
# This wheel is specific to Colab.

fa_installed = True

is_colab = True
if is_colab and not fa_installed:
    # Define the wheel details
    WHEEL_URL = "https://github.com/mjun0812/flash-attention-prebuild-wheels/releases/download/v0.4.11/flash_attn-2.8.3+cu126torch2.8-cp312-cp312-linux_x86_64.whl"
    WHEEL_NAME = "flash_attn-2.8.3+cu126torch2.8-cp312-cp312-linux_x86_64.whl"

    # Download and install the wheel
    !wget {WHEEL_URL}
    !pip install {WHEEL_NAME}

    # Clean up the downloaded file
    import os
    os.remove(WHEEL_NAME)

    print("\n✅ FlashAttention 2 installed successfully!")

## Weights & Biases

To provide your wandb API key for the script:
1. You could paste it in manually on the training command lines further down.
2. Or, use the secrets panel (the key symbol on the left edge of the notebook) and:
    * Define your wandb api key as `wandb_api_key`.
    * Grant access to this notebook.
    * Run the below cell to retrieve it.

In [ ]:
import os
# Set to false if you don't want to use wandb.
# The scripts will still log using the wandb library, but to a local directory.
use_wandb = True
is_colab = True
if use_wandb:

    # Enable Weights & Biases logging (online mode)
    wandb_mode = "online"

    os.environ["WANDB_MODE"] = "online"

    if is_colab:
        # Get key from colab secrets
        from google.colab import userdata

        # Get wandb API key from Colab secrets
        wandb_key = userdata.get("wandb_api_key")
    else:
        # Get the key from the environment
        # Import python dot env
        from dotenv import load_dotenv

        # Load the environment variables from the .env file
        load_dotenv()

        wandb_key = os.getenv("wandb_api_key")

# Set to offline if you don't want to log in.
else:
    wandb_mode = "offline"

    wandb_key = ""

In [ ]:
os.environ["WANDB_MODE"]

'online'

# ▂▂▂▂▂▂▂▂▂▂▂▂

# Model Source Code

# linkspace_config.py

In [ ]:
"""# `linkspace_config.py`

Configuration for LinkedSpaceDecoder model with flexible space-to-module mappings.

This config allows arbitrary mappings of modules to shared subspaces, enabling
experimentation with different architectural configurations.
"""

from typing import Optional, Dict, List, Any

import torch
from torch import nn

from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel

#### `make_shorthand`

In [ ]:
def make_shorthand(model_cfg):
    """
    Takes an instance of LinkedSpaceDecoderConfig and constructs a shorthand
    name for the model based on settings.
    """

    # Build a string representation of the spaces configuration
    space_strs = []
    for space_id, space_config in model_cfg.spaces.items():
        size = space_config['size']
        modules = ','.join(space_config['modules'])
        space_strs.append(f"sp{space_id}[{size}:{modules}]")

    spaces_str = " + ".join(space_strs)

    # Assemble string
    shorthand = (
        f"linkspace - {spaces_str} - "
        f"h{model_cfg.hidden_size} - l{model_cfg.num_hidden_layers}"
    )

    return shorthand

### `LinkedSpaceDecoderConfig`

In [ ]:
class LinkedSpaceDecoderConfig(PretrainedConfig):
    r"""
    Configuration class for LinkedSpaceDecoder.

    Extends the HuggingFace `PretrainedConfig` to support flexible space-to-module
    mappings. Instead of fixed projections for Q, K, V, O and FFN, this config allows
    arbitrary grouping of modules into shared subspaces.

    ----------------------
    Core Model Parameters:
    ----------------------
    - vocab_size (`int`) — Vocabulary size.
    - hidden_size (`int`) — Model hidden dimension.
    - num_hidden_layers (`int`) — Number of transformer blocks.
    - intermediate_size (`int`) — Feed-forward hidden dimension.
    - hidden_act (`str`) — Activation function.
    - hidden_dropout_prob (`float`) — Dropout after projections and FFNs.
    - attention_dropout_prob (`float`) — Dropout applied to attention scores.
    - max_position_embeddings (`int`) — Max sequence length.
    - initializer_range (`float`) — Stddev of weight init.

    - rms_norm_eps (`float`) — Epsilon for RMSNorm (all norms are RMSNorm)

    - classifier_dropout (`float` or None) — Dropout for final classifier.

    - vocab_subspace (`bool`) — Whether to decompose vocabulary embeddings
    - vocab_rank (`int`) — Rank of vocabulary subspace

    ----------------------------------
    LinkedSpace Architecture:
    ----------------------------------
    - spaces (`dict`) — Dictionary mapping space IDs to space configurations.
      Each space config has:
        - size (`int`) — Dimension of the shared subspace
        - modules (`list`) — List of module names using this space
          Valid module names:
            Attention: "Q", "K", "V", "O"
            FFN: "in", "gate", "out"

      All spaces use RMSNorm normalization (always enabled).

      Example:
        spaces = {
            0: {"size": 768, "modules": ["K", "V", "Q", "in", "gate", "out"]},
            1: {"size": 256, "modules": ["O"]}
        }

    - num_attention_heads (`int`) — Number of attention heads.
    - qk_private_dim (`int`) — Query/key private dimension per head.
    - vo_private_dim (`int`) — Value/output private dimension per head.

    - rope_dims (`int`) — Number of head dimensions carrying RoPE.
    - nope_dims (`int`) — Non-positional encoding dimensions.
    - rope_theta (`float`) — Base frequency used for RoPE.
    - rope_scaling (`dict` or None) — HF-style scaling dict for RoPE.
    - attention_bias (`bool`) — Whether to include bias terms in projections.

    - num_dense_layers (`int`) — Number of leading layers that do not use
                                 subspaces for attention or FFNs.
    - attention_backend (`str`) — Must be one of `"eager"`, `"flash_attention_2"`, or `"sdpa"`.
    """

    model_type = "linkspace_decoder"

    def __init__(
        self,

        # === Core Model ===
        vocab_size:         int = 30522,
        hidden_size:        int = 512,
        num_hidden_layers:  int = 12,

        shrd_intrmd_size:   int = 2304,
        dense_intrmd_size:  int = 3072,

        interleave_dense:   str = None,

        hidden_dropout_prob=0.1,
        attention_dropout_prob=0.1,
        max_position_embeddings: int = 2048,
        initializer_range=0.02,
        rms_norm_eps=1e-6,
        classifier_dropout=None,

        vocab_subspace=False,
        vocab_rank=None,
        tie_word_embeddings=True,

        # === LinkedSpace Configuration ===
        spaces: Optional[Dict[int, Dict[str, Any]]] = None,

        # === Attention Parameters ===
        shrd_attn_heads:    int = 12,
        dense_attn_heads:   int = 16,

        rope_dims:           int = 16,

        # Private head dimensions
        qk_private_dim:      int = None,
        vo_private_dim:      int = None,
        nope_dims:           int = None,

        attention_backend="eager",
        rope_theta=10000.0,
        rope_scaling=None,
        attention_bias=False,

        # === Layer Composition ===
        num_dense_layers=12,  # dense MHA layers before linkspace starts

        **kwargs
    ) -> None:
        super().__init__(**kwargs)

        # === Core Model ===
        self.vocab_size = vocab_size
        self.hidden_size = hidden_size

        self.num_hidden_layers = num_hidden_layers
        self.interleave_dense = interleave_dense

        self.dense_intrmd_size = dense_intrmd_size
        self.shrd_intrmd_size = shrd_intrmd_size

        self.hidden_dropout_prob = hidden_dropout_prob
        self.attention_dropout_prob = attention_dropout_prob
        self.max_position_embeddings = max_position_embeddings
        self.initializer_range = initializer_range
        self.rms_norm_eps = rms_norm_eps
        self.classifier_dropout = classifier_dropout

        self.vocab_subspace = vocab_subspace
        self.vocab_rank = vocab_rank
        self.tie_word_embeddings = tie_word_embeddings

        # === LinkedSpace Configuration ===
        # If no spaces provided, default to a simple configuration
        if spaces is None:
            spaces = {
                0: {"size": hidden_size, "modules": ["Q", "K", "V", "O", "in", "gate", "out"]}
            }
        self.spaces = spaces

        # === Attention ===
        self.dense_attn_heads = dense_attn_heads
        self.shrd_attn_heads = shrd_attn_heads

        self.rope_dims = rope_dims

        # Private head dimensions
        self.qk_private_dim = qk_private_dim
        self.vo_private_dim = vo_private_dim
        self.nope_dims = nope_dims
        self.rope_theta = rope_theta
        self.rope_scaling = rope_scaling
        self.attention_bias = attention_bias
        self.num_dense_layers = num_dense_layers

        # === Attention backend ===
        self.attention_backend = attention_backend

        # === Validation ===
        #self._validate()

        #print(f"  > LinkedSpace *Config.init: {make_shorthand(self)}\n")

    def _validate(self):
        """Validate the configuration."""

        # === Model ===
        if self.num_dense_layers > self.num_hidden_layers:
            raise ValueError("`num_dense_layers` must be <= `num_hidden_layers`")
        if self.vocab_subspace and self.vocab_rank is None:
            raise ValueError("`vocab_rank` must be set when `vocab_subspace=True`")

        # === LinkedSpace Validation ===
        valid_modules = {"Q", "K", "V", "O", "in", "gate", "out"}

        # Check that each space has required keys
        for space_id, space_config in self.spaces.items():
            if not isinstance(space_config, dict):
                raise ValueError(f"Space {space_id} config must be a dictionary")

            if 'size' not in space_config:
                raise ValueError(f"Space {space_id} must have 'size' key")
            if 'modules' not in space_config:
                raise ValueError(f"Space {space_id} must have 'modules' key")

            # Validate module names
            for module in space_config['modules']:
                if module not in valid_modules:
                    raise ValueError(
                        f"Invalid module '{module}' in space {space_id}. "
                        f"Valid modules: {valid_modules}"
                    )

        # Check that each module is assigned to exactly one space
        module_assignments = {}
        for space_id, space_config in self.spaces.items():
            for module in space_config['modules']:
                if module in module_assignments:
                    raise ValueError(
                        f"Module '{module}' is assigned to multiple spaces: "
                        f"{module_assignments[module]} and {space_id}"
                    )
                module_assignments[module] = space_id

        # Validate that private dimensions are set
        if self.qk_private_dim is None or self.vo_private_dim is None:
            raise ValueError("Must set qk_private_dim and vo_private_dim")
        if self.nope_dims is None:
            raise ValueError("Must set nope_dims")

        # === Attention Backend ===
        valid_backends = ["eager", "flash_attention_2", "sdpa"]
        if self.attention_backend not in valid_backends:
            raise ValueError(f"Unknown attention backend: {self.attention_backend}, options are {valid_backends}")

    def get_space_for_module(self, module: str) -> Optional[int]:
        """
        Get the space ID that a given module is assigned to.

        Args:
            module: Module name (e.g., "Q", "K", "V", "O", "in", "gate", "out")

        Returns:
            Space ID if module is in a space, None otherwise
        """
        for space_id, space_config in self.spaces.items():
            if module in space_config['modules']:
                return space_id
        return None


#### `get_config`

import json

#### `get_config`

In [ ]:
 def get_config(filename):
    """Load configuration from a JSON file."""

    # Load the config file.
    with open(filename) as f:
        full_cfg = json.load(f)

    # Strict key check on the model configuration.

    # Get the list of keys allowed / required by `*Config`
    valid_keys = LinkedSpaceDecoderConfig.__init__.__code__.co_varnames
    # Remove `self` and `kwargs`
    valid_keys = set(valid_keys) - {"self", "kwargs"}

    # Compare the set of keys in the json file vs `*Config`
    extra_keys = set(full_cfg["model"]) - valid_keys
    missing_keys = valid_keys - set(full_cfg["model"])

    # If there any in the `json` that aren't in `*Config`,
    if extra_keys:
        # List them for the user.
        raise ValueError(f"Unknown keys in config: {sorted(extra_keys)}")

    #  If the json config is missing required keys,
    if missing_keys:
        # List them for the user.
        raise ValueError(f"config json is missing: {sorted(missing_keys)}")

    # Will raise TypeError, by design, if required args are missing
    # The asterisks unpack the dictionary into a list of keywords as though
    # all of the settings were writting out individually.
    model_cfg = LinkedSpaceDecoderConfig(**full_cfg["model"])

    return full_cfg, model_cfg

# linkspace_layer.py

In [ ]:
"""# ▂▂▂▂▂▂▂▂▂▂▂▂

# `linkspace_layer.py`

LinkedSpace Decoder Layer combining attention and feedforward with flexible space-to-module mappings.

This unified layer allows attention outputs and FFN inputs to share the same space.
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional

#from models.linkspace_config import LinkedSpaceDecoderConfig

### `DeepseekV3RMSNorm`

In [ ]:
class DeepseekV3RMSNorm(nn.Module):
    def __init__(self, hidden_size, eps=1e-6):
        """
        DeepseekV3RMSNorm is equivalent to T5LayerNorm
        """
        super().__init__()
        self.weight = nn.Parameter(torch.ones(hidden_size))
        self.variance_epsilon = eps

    def forward(self, hidden_states):
        input_dtype = hidden_states.dtype
        hidden_states = hidden_states.to(torch.float32)
        variance = hidden_states.pow(2).mean(-1, keepdim=True)
        hidden_states = hidden_states * torch.rsqrt(variance + self.variance_epsilon)
        return self.weight * hidden_states.to(input_dtype)

## `RotaryEmbedding`

In [ ]:

class RotaryEmbedding(nn.Module):
    """Precompute RoPE embeddings and store them as buffers."""

    def __init__(self, config: LinkedSpaceDecoderConfig) -> None:
        super().__init__()

        dim = config.rope_dims
        seq_len = config.max_position_embeddings

        # ------------------------------
        # Compute inverse frequencies
        # ------------------------------
        # Shape: [dim // 2]
        #   inv_freq[i] = 1 / (theta^(i / dim))
        inv_freq = 1.0 / (
            config.rope_theta
            ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim)
        )

        # ------------------------------
        # Apply RoPE scaling if configured
        # ------------------------------
        if config.rope_scaling is not None:
            scaling_type = config.rope_scaling.get("type", "linear")
            scaling_factor = config.rope_scaling.get("factor", 1.0)

            if scaling_type == "linear":
                # Linear scaling: divide frequencies by scaling factor
                inv_freq = inv_freq / scaling_factor
            elif scaling_type == "dynamic":
                # Dynamic scaling: adjust based on sequence length
                # This is a simplified implementation
                inv_freq = inv_freq / scaling_factor
            else:
                print(f"Warning: Unknown RoPE scaling type '{scaling_type}', using linear scaling")
                inv_freq = inv_freq / scaling_factor

        # ------------------------------
        # Compute position indices
        # ------------------------------
        # Shape: [seq_len]
        t = torch.arange(seq_len, dtype=torch.float32)

        # ------------------------------
        # Outer product: [seq_len, dim // 2]
        # Each row i contains: t[i] * inv_freq
        # ------------------------------
        freqs = torch.outer(t, inv_freq)

        # ------------------------------
        # Duplicate for interleaved sin/cos: [seq_len, dim]
        # This matches the common format: [sin_0, cos_0, sin_1, cos_1, ...]
        # ------------------------------
        emb = torch.cat((freqs, freqs), dim=-1)

        # ------------------------------
        # Register cos/sin as buffers
        # - Stored in float32
        # - Will be moved to correct device/dtype via model.to(...)
        # - Not saved with state_dict (persistent=False)
        # ------------------------------
        self.register_buffer("cos", emb.cos(), persistent=False)
        self.register_buffer("sin", emb.sin(), persistent=False)

    def forward(self, position_ids: torch.LongTensor) -> tuple[torch.Tensor, torch.Tensor]:
        """ """
        return None # This function is not necessary.


#### `rotate_half`

In [ ]:
def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

## `LinkedSpaceDecoderLayer`

In [ ]:
class LinkedSpaceDecoderLayer(nn.Module):
    """
    A unified decoder layer combining attention and feed-forward networks.

    This layer merges the attention and FFN components to allow attention outputs
    and FFN inputs to share the same space, enabling more efficient parameter usage.

    Architecture:
        1. Pre-attention norm (RMSNorm on hidden_size)
        2. Multi-head latent attention with flexible space mappings for Q, K, V, O
        3. Residual connection after attention
        4. Pre-FFN norm (RMSNorm on hidden_size)
        5. SwiGLU feed-forward with flexible space mappings for in, gate, out
        6. Residual connection after FFN

    All normalization layers are RMSNorm and are always present.
    """

    def __init__(self, config: LinkedSpaceDecoderConfig, layer_idx: int, is_dense: bool):
        super().__init__()

        self.config = config
        self.layer_idx = layer_idx
        self.attention_dropout_prob = config.attention_dropout_prob

        # Attention parameters

        self.rope_theta = config.rope_theta
        self.rope_dims = config.rope_dims
        self.nope_dims = config.nope_dims
        self.qk_private_dim = config.qk_private_dim
        self.vo_private_dim = config.vo_private_dim
        self.hidden_size = config.hidden_size

        if is_dense:
            self.intermediate_size = config.dense_intrmd_size
            self.num_heads = config.dense_attn_heads
        else:
            self.intermediate_size = config.shrd_intrmd_size
            self.num_heads = config.shrd_attn_heads

        # Determine if this is a dense layer
        self.is_dense = is_dense

        print(f"init: Layer {layer_idx}, dense: {is_dense}")

        # =========================
        #   Normalization Layers
        # =========================
        # All norms are RMSNorm and always present
        self.attn_input_norm = DeepseekV3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.ffn_input_norm = DeepseekV3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        # =========================
        #   Setup Projections
        # =========================
        self._setup_unified_projections(config)

        # Softmax scaling factor
        self.softmax_scale = self.qk_private_dim ** (-0.5)

    def _setup_unified_projections(self, config: LinkedSpaceDecoderConfig):
        """
        Setup unified space-based projections for all modules (Q, K, V, O, in, gate, out).

        All modules are treated uniformly using the spaces architecture:
        1. Create a projection and norm for each unique shared space
        2. Build module -> space_id mapping for all modules
        3. Create private projections for each module
        4. Output modules (O, out) have special handling: use transposed space projections

        Dense layers use "identity" space; linkspace layers require all modules in spaces.
        """

        # =========================
        # Step 1: Create Space Projections
        # =========================

        # Store space projections and norms in ModuleDict
        self.space_projections = nn.ModuleDict()
        self.space_norms = nn.ModuleDict()

        if self.is_dense:
            # Dense layer: create identity space for all modules
            self.space_projections["identity"] = nn.Identity()
            self.space_norms["identity"] = nn.Identity()
            space_sizes = {"identity": config.hidden_size}
        else:
            # Linkspace layer: create projections for each defined space
            space_sizes = {}
            for space_id, space_config in config.spaces.items():
                space_key = str(space_id)
                space_size = space_config['size']
                space_sizes[space_key] = space_size

                # If space_size is -1, use identity (no subspace projection)
                if space_size == -1:
                    space_sizes[space_key] = config.hidden_size
                    self.space_projections[space_key] = nn.Identity()
                    self.space_norms[space_key] = nn.Identity()
                else:
                    space_sizes[space_key] = space_size

                    # Create projection WITHOUT bias (allows transposed reuse for outputs)
                    self.space_projections[space_key] = nn.Linear(
                        config.hidden_size,
                        space_size,
                        bias=False,
                    )

                    # Create RMSNorm (always present for linkspace)
                    self.space_norms[space_key] = DeepseekV3RMSNorm(
                        space_size,
                        eps=config.rms_norm_eps
                    )

        # =========================
        # Step 2: Build Module -> Space Mapping
        # =========================

        self.module_to_space = {}
        all_modules = ["Q", "K", "V", "O", "in", "gate", "out"]

        if self.is_dense:
            # All modules use identity space
            for module in all_modules:
                self.module_to_space[module] = "identity"
        else:
            # All modules must be in a defined space
            for module in all_modules:
                space_id = config.get_space_for_module(module)
                if space_id is None:
                    raise ValueError(
                        f"Module '{module}' must be assigned to a space in linkspace layers. "
                        f"Layer {self.layer_idx} is a linkspace layer (>= num_dense_layers)."
                    )
                self.module_to_space[module] = str(space_id)

        # =========================
        # Step 3: Attention Private Projections
        # =========================

        # Query
        q_space_key = self.module_to_space["Q"]
        q_input_dim = space_sizes[q_space_key]
        self.q_private_proj = nn.Linear(
            q_input_dim,
            self.num_heads * self.qk_private_dim,
            bias=False
        )

        # Key
        k_space_key = self.module_to_space["K"]
        k_input_dim = space_sizes[k_space_key]
        self.k_private_proj = nn.Linear(
            k_input_dim,
            self.num_heads * self.qk_private_dim,
            bias=False
        )

        # Value
        v_space_key = self.module_to_space["V"]
        v_input_dim = space_sizes[v_space_key]
        self.v_private_proj = nn.Linear(
            v_input_dim,
            self.num_heads * self.vo_private_dim,
            bias=False
        )

        # Output (O) - uses transposed space projection
        o_space_key = self.module_to_space["O"]
        o_space_dim = space_sizes[o_space_key]

        self.o_private_proj = nn.Linear(
            self.num_heads * self.vo_private_dim,
            o_space_dim,
            bias=False
        )

        # O's normalization (applied before transposed projection)
        # TODO - Don't define norm if there's no output subspace.
        if self.is_dense or (o_space_dim == config.hidden_size):
            self.o_norm = nn.Identity()
        else:
            self.o_norm = DeepseekV3RMSNorm(o_space_dim, eps=config.rms_norm_eps)

        # =========================
        # Step 4: FFN Private Projections
        # =========================

        # Input (in)
        in_space_key = self.module_to_space["in"]
        in_space_dim = space_sizes[in_space_key]
        self.in_private_proj = nn.Linear(
            in_space_dim,
            self.intermediate_size,
            bias=True
        )

        # Gate
        gate_space_key = self.module_to_space["gate"]
        gate_space_dim = space_sizes[gate_space_key]
        self.gate_private_proj = nn.Linear(
            gate_space_dim,
            self.intermediate_size,
            bias=True
        )

        # Output (out) - uses transposed space projection
        out_space_key = self.module_to_space["out"]
        out_space_dim = space_sizes[out_space_key]

        self.out_private_proj = nn.Linear(
            self.intermediate_size,
            out_space_dim,
            bias=False
        )

        # Out's normalization (applied before transposed projection)
        # TODO - Don't define norm if there's no output subspace.
        if self.is_dense or (out_space_dim == config.hidden_size):
            self.out_norm = nn.Identity()
        else:
            self.out_norm = DeepseekV3RMSNorm(out_space_dim, eps=config.rms_norm_eps)

    def forward(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor],
        **kwargs,
    ) -> torch.Tensor:
        """
        Forward pass through the unified decoder layer.

        Args:
            hidden_states: Input tensor [B, T, D]
            position_embeddings: Tuple of (cos, sin) RoPE embeddings
            attention_mask: Optional attention mask

        Returns:
            Output tensor [B, T, D] after attention + FFN with residuals
        """

        # === Tensor Dimension Symbols ===
        #    B: batch_size     — number of samples in the batch
        #    T: seq_len        — number of tokens per sample
        #    H: n_heads        — number of attention heads
        #    D: hidden_dim     — model embedding size
        #   Dq: qk_private_dim - per-head query/key dimension
        #   Dv: vo_private_dim - per-head value/output dimension
        #   Dr: rope_dims      - dimensions receiving RoPE
        #  Dff: intermediate_size - FFN hidden dimension

        # ========================
        #     Self Attention
        # ========================
        residual = hidden_states

        # Normalize the hidden states to create the input to attention
        attn_input = self.attn_input_norm(hidden_states)

        # Run attention
        attn_output = self._forward_attention(attn_input, position_embeddings, attention_mask)

        # Add residual connection
        hidden_states = residual + attn_output

        # ===========================
        #     Feed-Forward Network
        # ===========================
        residual = hidden_states

        # Normalize the updated hidden states prior to the FFN
        ffn_input = self.ffn_input_norm(hidden_states)

        # Run FFN
        ffn_output = self._forward_feedforward(ffn_input)

        # Add residual connection
        hidden_states = residual + ffn_output

        return hidden_states, attn_output, ffn_output

    def _forward_attention(
        self,
        hidden_states: torch.Tensor,
        position_embeddings: tuple[torch.Tensor, torch.Tensor],
        attention_mask: Optional[torch.Tensor],
    ) -> torch.Tensor:
        """
        Compute multi-head latent attention with flexible space mappings.

        Args:
            hidden_states: Normalized input [B, T, D]
            position_embeddings: Tuple of (cos, sin) RoPE embeddings
            attention_mask: Optional attention mask

        Returns:
            Attention output [B, T, D]
        """
        B, T = hidden_states.shape[:2]
        H = self.num_heads
        Dq = self.qk_private_dim
        Dv = self.vo_private_dim

        # ==============================
        #   Unified Space-Based Forward
        # ==============================

        # Cache for evaluated spaces to avoid redundant computation
        space_cache = {}

        def get_space_representation(module: str) -> torch.Tensor:
            """
            Get the space representation for a module (Q, K, V, O, in, gate, out).
            Uses caching to avoid recomputing the same space projection.
            """
            space_key = self.module_to_space[module]

            # Check if already computed
            if space_key not in space_cache:
                # Apply space projection and normalization
                space_proj  = self.space_projections[space_key]
                space_norm = self.space_norms[space_key]

                space_repr = space_proj(hidden_states)
                space_repr = space_norm(space_repr)

                # Cache for reuse
                space_cache[space_key] = space_repr

            return space_cache[space_key]

        # === Query ===
        q_space = get_space_representation("Q")
        queries = self.q_private_proj(q_space)

        # === Key ===
        k_space = get_space_representation("K")
        keys = self.k_private_proj(k_space)

        # === Value ===
        v_space = get_space_representation("V")
        values = self.v_private_proj(v_space)

        # Split up queries, keys, values for multi-head attention
        # Inputs:  Each [B, T, H*Dh]
        # Outputs: Each [B, H, T, Dh]
        queries = queries.view(B, T, H, Dq).transpose(1, 2)
        keys = keys.view(B, T, H, Dq).transpose(1, 2)
        values = values.view(B, T, H, Dv).transpose(1, 2)

        # ==================
        #        RoPE
        # ==================

        # 1. Unpack the precomputed cosine and sine embeddings
        cos, sin = position_embeddings

        # 2. Split the query and key heads into the part to rotate and the part to pass through
        q_rope, q_pass = queries[..., :self.rope_dims], queries[..., self.rope_dims:]
        k_rope, k_pass = keys[..., :self.rope_dims], keys[..., self.rope_dims:]

        # 3. Apply the rotary embedding to the designated slice
        # To broadcast cos and sin across the batch and head dimensions, we unsqueeze them.
        # Shape change: [T, Dr] -> [1, 1, T, Dr]
        cos = cos.unsqueeze(0).unsqueeze(0)
        sin = sin.unsqueeze(0).unsqueeze(0)

        q_rotated = (q_rope * cos) + (rotate_half(q_rope) * sin)
        k_rotated = (k_rope * cos) + (rotate_half(k_rope) * sin)

        # 4. Concatenate the rotated and pass-through parts back together
        queries = torch.cat((q_rotated, q_pass), dim=-1)
        keys = torch.cat((k_rotated, k_pass), dim=-1)

        # ===================
        #       Attention
        # ===================

        # Only apply dropout during training
        dropout_p = self.attention_dropout_prob if self.training else 0.0

        # Call SDPA / Flash Attention
        attn_output = F.scaled_dot_product_attention(
            queries,
            keys,
            values,
            attn_mask=None,
            dropout_p=dropout_p,
            scale=self.softmax_scale,
            is_causal=True,
        )

        # Reshape output back to [B, T, H * Dv] from [B, H, T, Dv]
        attn_output = attn_output.transpose(1, 2).contiguous().view(B, T, H * Dv)

        # =========================
        #  Unified Output Projection
        # =========================

        # Step 1: Project to output space (private projection)
        attn_output = self.o_private_proj(attn_output)


        # Step 3: Project back to hidden size using TRANSPOSED space projection
        o_space_key = self.module_to_space["O"]
        o_space_proj = self.space_projections[o_space_key]

        if isinstance(o_space_proj, nn.Identity):
            # Dense layer - identity projection
            attn_output = o_space_proj(attn_output)
        else:
            # Apply O's normalization (RMSNorm)
            attn_output = self.o_norm(attn_output)

            # Linkspace layer - use transposed weight to go from space back to hidden
            attn_output = F.linear(attn_output, o_space_proj.weight.t(), bias=None)

        return attn_output

    def _forward_feedforward(self, x: torch.Tensor) -> torch.Tensor:
        """
        Compute SwiGLU feed-forward network with unified space mappings.

        FFN(x) = W_out( Swish(W_in(x)) ⊙ W_gate(x) )

        Uses the same unified space architecture as attention:
        - Input modules (in, gate) go through: space_proj -> space_norm -> private_proj
        - Output module (out) goes through: private_proj -> out_norm -> transposed space_proj

        Args:
            x: Normalized input [B, T, D]

        Returns:
            FFN output [B, T, D]
        """

        # ==============================
        #   Unified Space-Based Forward
        # ==============================

        # Cache for evaluated spaces to avoid redundant computation
        # (can reuse if in and gate share a space)
        space_cache = {}

        def get_space_representation(module: str) -> torch.Tensor:
            """
            Get the space representation for a module.
            Uses caching to avoid recomputing the same space projection.
            """
            space_key = self.module_to_space[module]

            # Check if already computed
            if space_key not in space_cache:
                # Apply space projection and normalization
                space_proj = self.space_projections[space_key]
                space_norm = self.space_norms[space_key]

                space_repr = space_proj(x)
                space_repr = space_norm(space_repr)

                # Cache for reuse
                space_cache[space_key] = space_repr

            return space_cache[space_key]

        # === Input (in) ===
        in_space = get_space_representation("in")
        x_proj = self.in_private_proj(in_space)

        # === Gate ===
        gate_space = get_space_representation("gate")
        gate = self.gate_private_proj(gate_space)

        # SwiGLU nonlinearity
        ffn_hidden = F.silu(x_proj) * gate  # [B, T, intermediate_size]

        # =========================
        #  Output Projection (out)
        # =========================

        # Step 1: Project to output space (private projection)
        ffn_output = self.out_private_proj(ffn_hidden)

        # Step 3: Project back to hidden size using TRANSPOSED space projection
        out_space_key = self.module_to_space["out"]
        out_space_proj = self.space_projections[out_space_key]

        if isinstance(out_space_proj, nn.Identity):
            # Dense layer - identity projection
            ffn_output = out_space_proj(ffn_output)
        else:
            # Apply out's normalization (RMSNorm)
            ffn_output = self.out_norm(ffn_output)

            # Linkspace layer - use transposed weight to go from space back to hidden
            ffn_output = F.linear(ffn_output, out_space_proj.weight.t(), bias=None)

        return ffn_output

# linkspace_decoder.py

In [ ]:
# -*- coding: utf-8 -*-

"""# linkspace_decoder.py

LinkedSpaceDecoder model with flexible space-to-module mappings.
"""

from typing import Optional

import torch
from torch import nn

from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel
from transformers.modeling_attn_mask_utils import _prepare_4d_attention_mask_for_sdpa

#from layers.linkspace_mla import RotaryEmbedding
#from layers.linkspace_layer import LinkedSpaceDecoderLayer, DeepseekV3RMSNorm
#from models.linkspace_config import LinkedSpaceDecoderConfig

"""#### *PreTrainedModel"""

'#### *PreTrainedModel'

### `LinkedSpaceDecoderPreTrainedModel`

In [ ]:
class LinkedSpaceDecoderPreTrainedModel(PreTrainedModel):
    """
    The **PreTrainedModel object for LinkedSpaceDecoder.
    """

    config_class = LinkedSpaceDecoderConfig
    base_model_prefix = "model"

    def _init_weights(self, module: nn.Module) -> None:
        """Weight initialization hook used by :class:`PreTrainedModel`.

        ``PreTrainedModel.post_init`` will recursively apply this function to
        every submodule right after construction.
        """

        if isinstance(module, nn.Linear):
            # Standard linear layer initialization
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.bias is not None:
                module.bias.data.zero_()

        elif isinstance(module, nn.Embedding):
            # Initialize embeddings with normal distribution
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)
            if module.padding_idx is not None:
                module.weight.data[module.padding_idx].zero_()

        elif isinstance(module, DeepseekV3RMSNorm):
            # RMSNorm initialization: weight to 1.0, no bias term
            module.weight.data.fill_(1.0)

        elif isinstance(module, nn.LayerNorm):
            # LayerNorm initialization: bias to 0, weight to 1.0
            module.bias.data.zero_()
            module.weight.data.fill_(1.0)

"""# ▂▂▂▂▂▂▂▂▂▂▂▂

# Classes
"""

"""#### *Model"""

'#### *Model'

### `LinkedSpaceDecoderModel`

In [ ]:
class LinkedSpaceDecoderModel(LinkedSpaceDecoderPreTrainedModel):
    """
    The LinkedSpace decoder model (without language modeling head).

    Implements flexible space-to-module mappings for attention and FFN.
    """

    def __init__(self, config: LinkedSpaceDecoderConfig) -> None:
        super().__init__(config)

        # ============================
        #    Vocabulary Embeddings
        # ============================

        # If we're decomposing the token embeddings,
        if config.vocab_subspace:

            # Create the embedding table. Vocabulary embeddings are learned
            # in a lower dimensional latent space.
            self.vocab_embed = nn.Embedding(
                config.vocab_size, # Number of tokens
                config.vocab_rank  # Subspace dimension
            )

            # Selected token latents will be projected up to model size.
            # vocab_proj has shape [vocab_rank x model_size]
            self.vocab_proj = nn.Linear(
                config.vocab_rank,  # Size of latents
                config.hidden_size, # Model size
                bias=False
            )

        # Otherwise, for a dense vocabulary,
        else:
            # Create the dense embedding table in model space.
            self.vocab_embed = nn.Embedding(
                config.vocab_size,  # Number of tokens
                config.hidden_size  # Model size
            )

            self.vocab_proj = None

        # =====================
        #   RoPE Embeddings
        # =====================

        # Pre-computes the table of RoPE embeddings, leaving them in
        # GPU memory.
        self.rope = RotaryEmbedding(config)

        # ===================
        #    Create Layers
        # ===================

        layers = []

        # For each layer,
        for i in range(config.num_hidden_layers):

            # Set the first x layers as dense.
            if i < config.num_dense_layers:
                is_dense = True

            # Odd layers are dense
            elif config.interleave_dense == "dsds":
                is_dense = ((i + 1) % 2 == 1)

            # Even layers are dense
            elif config.interleave_dense == "sdsd":
                is_dense = ((i + 1) % 2 == 0)

            # If the 'interleave dense' string has been provided,
            elif len(config.interleave_dense) > 0:
                assert len(config.interleave_dense) == config.num_hidden_layers
                # Go based on the character at this layer position.
                is_dense = list(config.interleave_dense)[i] == "d"

            else:
                is_dense = False

            print(f"model: Layer {i}, dense: {is_dense}")

            #elif config.interleave_dense == "dssds":
            #    is_dense = i % 3 == 0

            # Create a **Layer, providing the config and indicating its number.
            layers.append(
                LinkedSpaceDecoderLayer(
                    config,
                    layer_idx = i,
                    is_dense = is_dense
                )
            )

        # Wrap in torch ModuleList
        self.layers = nn.ModuleList(layers)

        # Whatever huggingface does behind the scenes...
        self.post_init()


    def embed(self, input_ids: torch.LongTensor) -> torch.Tensor:
        """
        Return token embeddings for input ids.
        This will perform the up projection to model space if the vocabulary is
        decomposed.

        input_ids have shape [batch_size, seq_len]
        """

        # If the vocabulary is decomposed,
        if self.vocab_proj is not None:

            # Retrieve the latents
            #  input_ids: [batch_size, seq_len]
            #          x: [batch_size, seq_len, latent_dim]
            x = self.vocab_embed(input_ids)

            #  Project the latents back to model space and return.
            return(self.vocab_proj(x))

        # If the vocabulary is dense,
        else:
            # Just return the embeddings.
            return self.vocab_embed(input_ids)

    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
        **kwargs,
    ) -> torch.Tensor:
        """
        Run the full decoder stack with causal attention.

        Inputs:
            input_ids       [batch_size, seq_len]
            attention_mask  [batch_size, seq_len] - 1 for real tokens, 0 for padding

        Returns:
            Final decoder layer output   [batch_size, seq_len, model_size]
        """

        # Retrieve the token embeddings for this sequence.
        # These are model_size, regardless of whether the vocab is decompd.
        hidden_states = self.embed(input_ids)

        # Retrieve the rotary position embeddings for all of the positions in
        # our current input sequence.

        seq_len = hidden_states.size(1)

        # Retrieves just the ones necessary for the sequence length of the
        # input. These are vectors, two per token. Their length is the
        # number of head dimensions we're applying RoPE to.
        R_cos = self.rope.cos[:seq_len]
        R_sin = self.rope.sin[:seq_len]


        # ===============================
        #   Attention Mask Conversion
        # ===============================
        # Run the model!

        # For each decoder layer,

        if False:
            for layer_i in range(len(self.layers), 2):

                residual = hidden_states

                # Evaluate the next two layers
                hidden_states_1, attn_output_1, ffn_output_1 = self.layers[layer_i](
                    hidden_states,       # Token embeddings
                    (R_cos, R_sin),      # Rope embeddings, passed as a tuple.
                    None,      # Attn mask
                )

                hidden_states_2, attn_output_2, ffn_output_2 = self.layers[layer_i + 1](
                    hidden_states,       # Token embeddings
                    (R_cos, R_sin),      # Rope embeddings, passed as a tuple.
                    None,      # Attn mask
                )

                hidden_states = residual + attn_output_1 + ffn_output_1 + attn_output_2 + ffn_output_2


        if True:
            for layer_i, layer in enumerate(self.layers):

                # Evaluate the layer
                hidden_states, attn_output, ffn_output = layer(
                    hidden_states,       # Token embeddings
                    (R_cos, R_sin),      # Rope embeddings, passed as a tuple.
                    None,      # Attn mask
                )

        # Return the final output of the decoder stack.
        return hidden_states



# task_heads.py

In [ ]:

import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Optional, Union

from transformers.modeling_outputs import CausalLMOutputWithPast

#from models.linkspace_config import LinkedSpaceDecoderConfig
#from models.linkspace_decoder import (
#    LinkedSpaceDecoderPreTrainedModel,
#    LinkedSpaceDecoderModel,
#    DeepseekV3RMSNorm
#)

#### `create_norm_layer`

In [ ]:
def create_norm_layer(hidden_size: int, config: LinkedSpaceDecoderConfig) -> nn.Module:
    """
    Create a normalization layer based on the config norm_type.

    Args:
        hidden_size: The dimension to normalize over
        config: Configuration containing norm_type and epsilon values

    Returns:
        Either a LayerNorm or RMSNorm layer
    """
    if config.norm_type == "layernorm":
        return nn.LayerNorm(hidden_size, eps=config.layer_norm_eps)
    elif config.norm_type == "rmsnorm":
        #from models.linkspace_decoder import DeepseekV3RMSNorm
        return DeepseekV3RMSNorm(hidden_size, eps=config.rms_norm_eps)
    else:
        # This should be caught by config validation, but being defensive
        raise ValueError(f"Unknown norm_type: {config.norm_type}")

### `LinkedSpaceDecoderForCausalLM`

In [ ]:
class LinkedSpaceDecoderForCausalLM(LinkedSpaceDecoderPreTrainedModel):
    """
    LinkedSpace Decoder model with a causal language modeling head.

    This model extends the LinkedSpaceDecoderModel with:
    - A language modeling head that projects hidden states to vocabulary logits
    - Support for computing cross-entropy loss for language modeling
    - Proper HuggingFace compatibility for causal language modeling tasks
    - Decoder-specific initialization strategies

    The model can be used for:
    - Text generation
    - Language modeling pretraining
    - Fine-tuning on downstream tasks
    """

    def __init__(self, config: LinkedSpaceDecoderConfig) -> None:
        super().__init__(config)

        # Initialize the base decoder model
        self.model = LinkedSpaceDecoderModel(config)

        # Final layer norm before the language modeling head
        self.norm = DeepseekV3RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

        # Language modeling head
        # Projects from hidden_size to vocab_size to get logits for each token
        self.lm_head = nn.Linear(
            config.hidden_size,
            config.vocab_size,
            bias=False  # Following common practice in modern LMs
        )

        # Initialize weights with decoder-specific strategy
        # Note: tie_weights() will be called automatically by post_init() if config.tie_word_embeddings=True
        self.post_init()

    def _init_weights(self, module: nn.Module) -> None:
        """
        Decoder-specific weight initialization with special handling for language modeling head.

        Key differences from encoder initialization:
        - Language modeling head gets specialized initialization for stability
        - Weight tying considerations for embedding/lm_head relationship
        """

        # Use the base class initialization for most modules
        super()._init_weights(module)

        # Special handling for language modeling head
        if module is self.lm_head:
            # Use smaller initialization for the language modeling head
            # This helps with training stability in autoregressive generation
            # Common practice is to use std=initializer_range or smaller
            module.weight.data.normal_(mean=0.0, std=self.config.initializer_range)

            # If weight tying is not used, we might want even smaller init
            if self.model.vocab_proj is not None:
                # For vocab subspace models where weights aren't tied,
                # use a smaller scale to prevent initial logits from being too large
                module.weight.data.normal_(mean=0.0, std=self.config.initializer_range * 0.5)

    def get_input_embeddings(self):
        """Return the input embedding layer for compatibility with HuggingFace."""
        return self.model.vocab_embed

    def set_input_embeddings(self, value):
        """Set the input embedding layer for compatibility with HuggingFace."""
        self.model.vocab_embed = value

    def get_output_embeddings(self):
        """Return the output embedding layer (lm_head) for compatibility."""
        return self.lm_head

    def set_output_embeddings(self, new_embeddings):
        """Set the output embedding layer for compatibility."""
        self.lm_head = new_embeddings

    def tie_weights(self):
        """
        Tie the input and output embedding weights.

        This method sets the language modeling head's weight to be the same as
        the input embedding weight. This reduces the number of parameters and
        is a common practice in modern language models.

        Note: For vocab subspace models, we need to handle the case where
        input embeddings go through a projection layer.
        """
        # Only tie when embeddings live in model space (no vocab_proj)
        if getattr(self.model, "vocab_proj", None) is None:
            # Use HF utility for correct tying/cloning semantics
            self._tie_or_clone_weights(self.lm_head, self.model.vocab_embed)
        # else: leave untied for subspace case


    def forward(
        self,
        input_ids: torch.LongTensor,
        attention_mask: Optional[torch.Tensor] = None,
        labels: Optional[torch.LongTensor] = None,
        **kwargs,
    ) -> Union[CausalLMOutputWithPast, tuple]:
        """
        Forward pass for causal language modeling.

        Args:
            input_ids: Token ids of shape [batch_size, seq_len]
            attention_mask: Attention mask of shape [batch_size, seq_len]
                           (1 for real tokens, 0 for padding)
            labels: Ground truth token ids for computing loss. Same shape as input_ids.
                   If provided, loss will be computed. Typically input_ids shifted by 1.

        Returns:
            CausalLMOutputWithPast containing:
            - logits: Prediction logits of shape [batch_size, seq_len, vocab_size]
            - loss: Cross-entropy loss if labels provided, else None
            - hidden_states: Final layer hidden states [batch_size, seq_len, hidden_size]
        """

        # Run the base decoder model
        # This applies all the transformer layers with causal attention
        hidden_states = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            **kwargs
        )

        # Apply final layer normalization
        # This normalizes the final hidden states before the language modeling head
        hidden_states = self.norm(hidden_states)

        # Project to vocabulary logits
        # Shape: [batch_size, seq_len, vocab_size]
        logits = self.lm_head(hidden_states)

        # Compute loss if labels are provided
        # Previously, we had custom loss computation here, but now we use the
        # standard HuggingFace loss function.
        loss = None
        if labels is not None:
            # Flatten the tokens
            loss = self.loss_function(
                logits,
                labels,
                vocab_size=self.config.vocab_size,
                **kwargs,
            )

        # Return in HuggingFace format
        return CausalLMOutputWithPast(
            loss=loss,
            logits=logits,
            past_key_values=None,  # Not implementing KV cache yet
            #hidden_states=hidden_states,
            hidden_states=hidden_states if kwargs.get("output_hidden_states", False) else None,
            attentions=None,
        )



# utils.py

In [ ]:
# `utils.py`

# Utility helpers for LinkedSpace experiments

from typing import Iterable, Tuple
import torch.nn as nn

#### `format_size`

In [ ]:
def format_size(num: int) -> str:
    """Return a human readable string for the given integer."""
    suffixes = [" ", "K", "M", "B"]
    base = 1024
    for suffix in suffixes:
        if abs(num) < base:
            if num % 1 != 0:
                return f"{num:.2f}{suffix}"
            else:
                return f"{num:.0f}{suffix}"
        num /= base
    if num % 1 != 0:
        return f"{num:.2f}T"
    return f"{num:.0f}T"

#### `summarize_parameters`

In [ ]:
def summarize_parameters(model: nn.Module, display_bias: bool = True) -> int:
    """Print a table of parameter names, shapes and counts."""
    params: Iterable[Tuple[str, nn.Parameter]] = list(model.named_parameters())
    print("The model has {:} different named parameters.\n".format(len(params)))

    total_params = 0
    for _, p in params:
        total_params += p.numel()

    print(f"Total elements: {format_size(total_params)}\n")
    print(
        "Parameter Name                                              Dimensions       Total Values    Trainable\n"
    )

    for p_name, p in params:
        p_size = list(p.size())
        for i in range(len(p_size) - 1, -1, -1):
            if p_size[i] == 1:
                del p_size[i]
        if len(p_size) == 1:
            if not display_bias:
                continue
            p_dims = "{:>10,} x {:<10}".format(p.size()[0], "-")
        elif len(p_size) == 2:
            p_dims = "{:>10,} x {:<10,}".format(p.size()[0], p.size()[1])
        elif len(p_size) == 3:
            p_dims = "{:>10,} x {:,} x {:<10}".format(p.size()[0], p.size()[1], p.size()[2])
        elif len(p_size) == 4:
            p_dims = "{:>10,} x {:,} x {:,} x {:<10}".format(
                p.size()[0], p.size()[1], p.size()[2], p.size()[3]
            )
        else:
            print("Unexpected: ", p.size(), p_name)
            break
        print(
            "{:<55} {:}    {:>6}    {:}".format(
                p_name, p_dims, format_size(p.numel()), p.requires_grad
            )
        )

    print(f"\nTotal elements: {format_size(total_params)}\n")
    return total_params

"""`def make_shorthand`"""

'`def make_shorthand`'

#### `make_shorthand`

In [ ]:
def make_shorthand(model_cfg):
    """
    Takes an instance of LinkedSpaceDecoderConfig and constructs a shorthand
    name for the model based on settings.
    """

    # Build a string representation of the spaces configuration
    space_strs = []
    for space_id, space_config in model_cfg.spaces.items():
        size = space_config['size']
        modules = ','.join(space_config['modules'])
        space_strs.append(f"sp{space_id}[{size}:{modules}]")

    spaces_str = " + ".join(space_strs)

    # Assemble string
    shorthand = (
        f"linkspace - {spaces_str} - "
        f"h{model_cfg.hidden_size} - l{model_cfg.num_hidden_layers}"
    )

    return shorthand

# ▂▂▂▂▂▂▂▂▂▂▂▂

# train.py

### Imports and Helpers

In [ ]:
# -*- coding: utf-8 -*-
# Updated training script for DeepSeek V3 with attention output subspace

"""# subspace_decoder/scripts/train.py"""


import os
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["USE_TF"] = "0"              # older check some codepaths still honor
# Optional: if Keras 3 is on the system and ever gets touched, force non-TF backend
os.environ.setdefault("KERAS_BACKEND", "torch")

from transformers.utils import is_tf_available
print("TF available (Transformers thinks):", is_tf_available())  # should be False


print("Importing Packages...\n")

import argparse
import json
import os
import shutil
import sys
from pathlib import Path

import torch
import wandb
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    TrainerCallback,
    set_seed,
)

#from utils import summarize_parameters, format_size
# To disable a warning.
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Make sure we can import modules from the decoder package
#PROJECT_ROOT = Path(__file__).resolve().parents[1]

#print("PROJECT_ROOT", PROJECT_ROOT)

#if str(PROJECT_ROOT) not in sys.path:
#    sys.path.insert(0, str(PROJECT_ROOT))

#from models.shared_space_config import SharedSpaceDecoderConfig, get_config
#from layers.task_heads import SharedSpaceDecoderForCausalLM

import torch.nn as nn

TF available (Transformers thinks): True
Importing Packages...



In [ ]:
def check_bf16_support():
    """Check if BFloat16 is supported on the current hardware and PyTorch version."""
    if not torch.cuda.is_available():
        print("Warning: CUDA not available. BFloat16 training requires CUDA.")
        return False

    # Check if the GPU supports BFloat16
    if hasattr(torch.cuda, 'is_bf16_supported') and torch.cuda.is_bf16_supported():
        print("✓ BFloat16 is supported on this hardware")
        return True

    # Fallback check for older PyTorch versions
    try:
        # Try to create a small BFloat16 tensor on GPU
        test_tensor = torch.tensor([1.0], dtype=torch.bfloat16, device='cuda')
        print("✓ BFloat16 is supported on this hardware")
        return True
    except Exception as e:
        print(f"Warning: BFloat16 not supported on this hardware: {e}")
        return False

In [ ]:
"""
def parse_args():
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True, help="Path to JSON config")
    return parser.parse_args()
"""

'\ndef parse_args():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--config", required=True, help="Path to JSON config")\n    return parser.parse_args()\n'

### Load Config

In [ ]:

"""Run pre-training using the provided configuration path."""
#config_path = "first_test.json"
#config_path = "second_test.json"
#config_path = "third_test.json"
#config_path = "fourth_test.json"
#config_path = "fifth_test.json"
#config_path = "ninth_test.json"
#config_path = "kv_test_1.json"
#config_path = "kv_test_2.json"
#config_path = "intrlv_test_1.json"
#config_path = "intrlv_test_3.json"
#config_path = "debug_test_1.json"
#config_path = "kv_test_2b.json"
#config_path = "link_oingate_1.json"
#config_path = "parallel_test_1.json"
#config_path = "intrlv_test_4.json"
#config_path = "intrlv_test_5.json"
#config_path = "intrlv_test_6.json"
#config_path = "intrlv_test_7.json"
#config_path = "intrlv_test_8.json"
config_path = "tiny-scale_baseline_wiki.json"
config_path = "gpt2_linkspace3.json"
#config_path = "gpt2_baseline2.json"
config_path = "gpt2_linkspace4.json"
config_path = "gpt2_linkspace5.json"

# Load configuration
full_cfg, model_cfg = get_config(config_path)

ptrain_cfg = full_cfg['pre_train']

# Print out its shorthand name.
print(full_cfg["shorthand"])

# Initialize the optional stats dictionary so later assignments don't fail.
if "stats" not in full_cfg:
    full_cfg["stats"] = {}

# Validate mixed precision settings
if ptrain_cfg["bf16"] and ptrain_cfg["fp16"]:
    raise ValueError("Cannot enable both bf16 and fp16 simultaneously. Please choose one.")

# Check BFloat16 compatibility if enabled
if ptrain_cfg["bf16"]:
    if not check_bf16_support():
        print("BFloat16 requested but not supported. Falling back to FP16.")
        ptrain_cfg["bf16"] = False
        ptrain_cfg["fp16"] = True

# Display torch.compile status
if ptrain_cfg["torch_compile"]:
    print(f"✓ torch.compile enabled:")
    print(f"  Backend: {ptrain_cfg['torch_compile_backend']}")
    print(f"  Mode: {ptrain_cfg['torch_compile_mode']}")
    print("  Note: First training step will be slower due to compilation.")
else:
    print("torch.compile disabled. Enable with 'torch_compile': true in config.")


tokenizer = AutoTokenizer.from_pretrained("gpt2")
# gpt2 has no pad by default; use EOS for padding in causal LM
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

# Verify vocab size matches
assert model_cfg.vocab_size == tokenizer.vocab_size

# Set random seed for reproducibility
set_seed(ptrain_cfg["seed"])

# Setup Weights & Biases
#if "WANDB_MODE" not in os.environ:
#    os.environ["WANDB_MODE"] = "offline"

wandb_api_key = os.environ.get("WANDB_API_KEY")

if wandb_api_key:
    wandb.login(key=wandb_api_key)


22l - 5h - kv192.ingt256 - mlp832
✓ BFloat16 is supported on this hardware
✓ torch.compile enabled:
  Backend: inductor
  Mode: default
  Note: First training step will be slower due to compilation.


### Dataset Prep

In [ ]:
# ======================
#    Load Dataset
# ======================

dataset_name = ptrain_cfg["dataset_name"]
dataset_config = ptrain_cfg["dataset_config"]

# Check if we should load a pre-processed dataset
if "preprocessed_dataset_path" in ptrain_cfg and ptrain_cfg["preprocessed_dataset_path"]:
    print(f"Loading pre-processed dataset from: {ptrain_cfg['preprocessed_dataset_path']}")
    from datasets import load_from_disk

    dataset = load_from_disk(ptrain_cfg["preprocessed_dataset_path"])
    print(f"Loaded pre-processed dataset:")
    print(f"  Train: {len(dataset['train']):,} examples")
    print(f"  Validation: {len(dataset['validation']):,} examples")

    # Skip tokenization and chunking since it's already done
    tokenized = dataset

elif dataset_name == "wikitext":

    # Original logic for wikitext and other datasets
    dataset = load_dataset(dataset_name, dataset_config)
elif dataset_name == "allenai/c4":
    raise ValueError(f"allenai/c4 requires prep-processing, but no preprocessed dataset path was provided")

else:
    raise ValueError(f"Dataset {dataset_name} not supported")

print(dataset)

# ========================
#    Tokenize Wikitext
# ========================

if dataset_name == "wikitext":

    block_size = ptrain_cfg["max_seq_length"]
    eos_id = tokenizer.eos_token_id

    # 1) Tokenize without truncation/padding
    def tokenize_function(examples):
        # add_special_tokens=False keeps things raw; we'll insert EOS between docs
        return tokenizer(
            examples["text"],
            add_special_tokens=False,
        )

    # 2) Group into contiguous blocks (concat + chunk)
    def group_texts(examples):
        # Flatten and insert EOS between documents to avoid cross-article bleed
        input_ids = []
        for ids in examples["input_ids"]:
            if len(ids) > 0:
                input_ids.extend(ids)
            # add an EOS fencepost between docs
            input_ids.append(eos_id)

        # Drop the trailing partial block so every example is full length
        total_length = (len(input_ids) // block_size) * block_size
        input_ids = input_ids[:total_length]

        # Split into equal blocks
        result_input_ids = [input_ids[i:i + block_size] for i in range(0, total_length, block_size)]
        # Labels are next-token targets; Trainer/model will do the shift
        return {
            "input_ids": result_input_ids,
            "labels": [ids.copy() for ids in result_input_ids],
            # Optional attention masks (all ones because no padding)
            "attention_mask": [[1] * block_size for _ in result_input_ids],
        }

    # Tokenize
    tokenized = dataset.map(
        tokenize_function,
        batched=True,
        num_proc=8,
        remove_columns=dataset["train"].column_names,  # drop raw "text"
    )

    # Concatenate + chunk
    tokenized = tokenized.map(
        group_texts,
        batched=True,
        num_proc=8,
    )



DatasetDict({
    test: Dataset({
        features: ['text'],
        num_rows: 4358
    })
    train: Dataset({
        features: ['text'],
        num_rows: 1801350
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 3760
    })
})


### Load Model & Train

In [ ]:
# Use a simple collator; we already created labels and have no pads
from transformers import default_data_collator
data_collator = default_data_collator

# ========================
#    Initialize Model
# ========================

print("Initializing model...")

model = LinkedSpaceDecoderForCausalLM(model_cfg)


Initializing model...
model: Layer 0, dense: True
init: Layer 0, dense: True
model: Layer 1, dense: False
init: Layer 1, dense: False
model: Layer 2, dense: False
init: Layer 2, dense: False
model: Layer 3, dense: True
init: Layer 3, dense: True
model: Layer 4, dense: False
init: Layer 4, dense: False
model: Layer 5, dense: False
init: Layer 5, dense: False
model: Layer 6, dense: True
init: Layer 6, dense: True
model: Layer 7, dense: False
init: Layer 7, dense: False
model: Layer 8, dense: False
init: Layer 8, dense: False
model: Layer 9, dense: True
init: Layer 9, dense: True
model: Layer 10, dense: False
init: Layer 10, dense: False
model: Layer 11, dense: False
init: Layer 11, dense: False
model: Layer 12, dense: True
init: Layer 12, dense: True
model: Layer 13, dense: False
init: Layer 13, dense: False
model: Layer 14, dense: False
init: Layer 14, dense: False
model: Layer 15, dense: True
init: Layer 15, dense: True
model: Layer 16, dense: False
init: Layer 16, dense: False
model: 

In [ ]:

# ================================
#       Review Configuration
# ================================

# Display architecture
print(model)

print("\n======== Model ========")
print(model_cfg)

print("\n======== Pre-Train ========")
print(json.dumps(ptrain_cfg, indent=2))

# Calculate and display effective batch size
device_batch_size = ptrain_cfg["train_batch_size"]
gradient_accumulation_steps = ptrain_cfg["gradient_accumulation_steps"]
effective_batch_size = device_batch_size * gradient_accumulation_steps

print(f"\n======== Batch Size Configuration ========")
print(f"Device batch size: {device_batch_size}")
print(f"Gradient accumulation steps: {gradient_accumulation_steps}")
print(f"Effective batch size: {effective_batch_size}")

print("=============================\n")

"""## Parameter Summary"""

print("\n======== Parameters ========")

## Get all of the model's parameters as a list of tuples.
params = list(model.named_parameters())

print('The model has {:} different named parameters.\n'.format(len(params)))

total_params = 0
for p_name, p in params:
    total_params += p.numel()

full_cfg["stats"]["total_elements"] = format_size(total_params)

print(f"Total elements: {full_cfg['stats']['total_elements']}\n")

# Display a full parameter breakdown using the shared utility
summarize_parameters(model)

# ========================================
#   Format Settings for WandB Run Name
# ========================================

# Format the cfg learning rate as a scientific notation string like 5e-4
lr_str = '{:.0e}'.format(ptrain_cfg['learning_rate'])

# Attention configuration

ptrain_cfg["run_name"] = full_cfg["stats"]["total_elements"] + " - " + full_cfg["shorthand"]

print(ptrain_cfg["run_name"])

"""## wandb and TrainingArguments"""

wandb.init(
    project=ptrain_cfg["wandb_project"],
    name=ptrain_cfg["run_name"],
    config=full_cfg
)

# ===============================
#       Training Arguments
# ===============================

training_args = TrainingArguments(
    output_dir=ptrain_cfg["output_dir"],

    per_device_train_batch_size=ptrain_cfg["train_batch_size"],
    per_device_eval_batch_size=ptrain_cfg["eval_batch_size"],
    gradient_accumulation_steps=ptrain_cfg["gradient_accumulation_steps"],

    bf16=ptrain_cfg["bf16"],
    fp16=ptrain_cfg["fp16"],

    # torch.compile configuration for performance optimization
    torch_compile=ptrain_cfg["torch_compile"],
    torch_compile_backend=ptrain_cfg["torch_compile_backend"],
    torch_compile_mode=ptrain_cfg["torch_compile_mode"],

    learning_rate=ptrain_cfg["learning_rate"],
    max_steps=ptrain_cfg["num_train_steps"],

    # TODO - Added this to recent 576 runs, but need to decide if it's needed.
    #max_grad_norm = 1.0,

    # The dataloader is a bottleneck without these.
    dataloader_num_workers=ptrain_cfg["num_workers"],
    dataloader_pin_memory=ptrain_cfg["pin_memory"],
    # The prefetch factor didn't appear to help.
    #dataloader_prefetch_factor = ptrain_cfg["prefetch_factor"],

    weight_decay=ptrain_cfg["weight_decay"],

    # Learning rate warmup (10% of total steps)
    warmup_steps=int(0.1 * ptrain_cfg["num_train_steps"]),
    lr_scheduler_type="linear",  # Linear warmup then decay

    # Evaluate every 2,000 steps
    # Note: Recent versions of Trainer changed the name from
    # `evaluation_strategy` to `eval_strategy`.
    batch_eval_metrics = True, # To avoid OOM
    eval_strategy="steps",
    eval_steps=ptrain_cfg["eval_steps"],
    eval_accumulation_steps=4,  # Process eval in smaller chunks to save memory

    logging_steps=ptrain_cfg["logging_steps"],
    metric_for_best_model="eval_loss",
    save_steps=ptrain_cfg["save_steps"],
    save_total_limit=2,           # Optional: keeps last 2 checkpoints
    save_strategy="steps",
    report_to=["wandb"],

    save_safetensors=False,

    run_name=ptrain_cfg["run_name"],

    remove_unused_columns=False,  # Optional: avoid dropping custom model inputs
)

# Print out all of the settings in TrainingArguments using tabulate.
# Note that they are not in alphabetical order, but that the order appears
# to be more sensible than that. (e.g., all batch size related arguments
# are together).
import tabulate
print("Training Arguments:")
print(tabulate.tabulate(vars(training_args).items(), headers=["Argument", "Value"]))

# ==========================
#     Perplexity Metric
# ==========================

import numpy as np

class PerplexityMetric:
    """
    A stateful class to compute perplexity in a batch-wise manner to avoid OOM.
    Similar to the MLMAccuracyMetric from the encoder training.
    """
    def __init__(self):
        # Initialize state variables to store running totals
        self.total_loss = 0.0
        self.total_tokens = 0

    def __call__(self, eval_pred, compute_result=False):
        """
        This method will be called by the Trainer.
        """
        predictions, labels = eval_pred

        # For causal LM, we compute perplexity
        # Shift predictions and labels for next token prediction
        shift_logits = predictions[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        # Flatten the tokens
        shift_logits = shift_logits.view(-1, shift_logits.size(-1))
        shift_labels = shift_labels.view(-1)

        # Create a mask for valid tokens (not padding, typically -100)
        mask = shift_labels != -100

        if mask.sum() > 0:  # Only compute if there are valid tokens
            # Compute loss only on valid tokens
            loss_fct = torch.nn.CrossEntropyLoss(reduction='sum')
            batch_loss = loss_fct(shift_logits[mask], shift_labels[mask])

            # Add to running totals
            self.total_loss += batch_loss.item()
            self.total_tokens += mask.sum().item()

        # If this is the final call after all batches are processed
        if compute_result:
            # Avoid division by zero
            if self.total_tokens == 0:
                avg_loss = 0.0
                perplexity = float('inf')
            else:
                avg_loss = self.total_loss / self.total_tokens
                perplexity = np.exp(avg_loss)

            # Prepare the final metrics dictionary
            metrics = {
                "perplexity": perplexity,
                "loss": avg_loss,
            }

            # Reset state for the next evaluation run
            self.total_loss = 0.0
            self.total_tokens = 0

            return metrics

        # For intermediate calls, return an empty dict
        return {}

# Instantiate your stateful metric computer
perplexity_metric = PerplexityMetric()

# ===============================
#           Trainer
# ===============================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized["train"],
    eval_dataset=tokenized["validation"],
    compute_metrics=perplexity_metric,

    # New argument, allows for other modalities.
    processing_class=tokenizer,

    data_collator=data_collator,
)

"""## Loop"""

# =====================
#     Run Training
# =====================

# Do inside a try/finally so that if the run aborts, we still call wandb.finish().
try:
    trainer.train()

    metrics = trainer.evaluate()

    wandb.log(metrics)

    # Store wandb ids into the config.
    full_cfg["pre_train"]["run_id"] = wandb.run.id
    full_cfg["pre_train"]["run_url"] = wandb.run.url
    full_cfg["pre_train"]["run_name"] = wandb.run.name

    # Save the best checkpoint.
    full_cfg["pre_train"]["best_checkpoint"] = trainer.state.best_model_checkpoint

    # Save the json back to disk
    with open(ptrain_cfg["output_dir"] + "/full_config.json", "w") as f:
        json.dump(full_cfg, f, indent=2)


finally:
    # End the wandb run.
    wandb.finish()


#if __name__ == "__main__":
#    args = parse_args()
#    main(args.config)

LinkedSpaceDecoderForCausalLM(
  (model): LinkedSpaceDecoderModel(
    (vocab_embed): Embedding(50257, 768)
    (rope): RotaryEmbedding()
    (layers): ModuleList(
      (0): LinkedSpaceDecoderLayer(
        (attn_input_norm): DeepseekV3RMSNorm()
        (ffn_input_norm): DeepseekV3RMSNorm()
        (space_projections): ModuleDict(
          (identity): Identity()
        )
        (space_norms): ModuleDict(
          (identity): Identity()
        )
        (q_private_proj): Linear(in_features=768, out_features=768, bias=False)
        (k_private_proj): Linear(in_features=768, out_features=768, bias=False)
        (v_private_proj): Linear(in_features=768, out_features=768, bias=False)
        (o_private_proj): Linear(in_features=768, out_features=768, bias=False)
        (o_norm): Identity()
        (in_private_proj): Linear(in_features=768, out_features=2048, bias=True)
        (gate_private_proj): Linear(in_features=768, out_features=2048, bias=True)
        (out_private_proj): Line

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 50256, 'bos_token_id': 50256, 'pad_token_id': 50256}.


Training Arguments:
Argument                                 Value
---------------------------------------  ----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------
output_dir                               checkpoints/link_22l_5h_kv192.ingt256_mlp832_wiki
overwrite_output_dir                     False
do_train                                 False
do_eval                                  True
do_predict                               False
eval_strategy                            IntervalStrategy.STEPS
prediction_loss_only                     False
per_device_train_batch_size              512
per_device_eval_batch_size               256
per_gpu_train_batch_size
per_gpu_eval_batch_size
gradient_accumulation_steps              2
eval_accumulation_steps                  4
eval_delay                               0
torch_empty_cache_steps
learning_rate       

# ▂▂▂▂▂▂▂▂▂▂▂▂

# (Ignore Below Here)

## Stashed Results

**GPT-2 Baseline**

```
 [ 390/3300 20:41 < 2:35:12, 0.31 it/s, Epoch 1.71/15]
Step	Training Loss	Validation Loss	Perplexity
300	4.782900	4.709316	110.976260
```

seqlen_128

```
 [ 333/3300 02:34 < 23:04, 2.14 it/s, Epoch 0.18/2]
Step	Training Loss	Validation Loss	Perplexity
300	5.116300	5.063547	158.150433
```

----

**Interleave Test 3 / "Fewer Params**

This one's misnamed. It did really well

```
deepspeed_plugin
 [2121/3300 11:52 < 06:36, 2.97 it/s, Epoch 2.34/4]
Step	Training Loss	Validation Loss	Perplexity
150	7.541100	6.987664	1083.187826
300	5.877800	5.696788	297.909035
450	5.248200	5.106646	165.115661
600	4.906300	4.784043	119.586916
750	4.674900	4.576163	97.140942
900	4.512900	4.427003	83.680212
1050	4.368900	4.296827	73.466335
1200	4.250100	4.173937	64.970754
1350	4.168000	4.087836	59.610764
1500	4.089900	4.023022	55.869684
1650	4.049400	3.969225	52.943492
1800	3.996600	3.924318	50.618533
1950	3.949100	3.886178	48.724323
2100	3.906600	3.857968	47.369017
```



**Interleave Test 2**

This one failed--I'd have to go look at it again. Did I try running sparse-dense, and maybe there's a bug in that arrangement?

**Interleave Test 1**

* Shared KV, shared ingate
* No Q, O
* Yes out
* 8 layers (6 heads, intmd ~512)
    * intmd 512 --> 520 to balance

No, worse

```
deepspeed_plugin
 [1682/3300 08:51 < 08:32, 3.16 it/s, Epoch 1.85/4]
Step	Training Loss	Validation Loss	Perplexity
300	5.899100	5.716613	303.873922
600	4.909900	4.790148	120.319191
900	4.516600	4.430273	83.954344
1200	4.278100	4.198760	66.603713
1500	4.101300	4.031440	56.342012
```

**KV Test 2**

* Sharing KV, nothing on FFNs
* Balanced with 7 layers, 7 heads, intermediate 544.
* Kinda large KV? 192

Very close to test 1, but slightly worse.

```
deepspeed_plugin
 [1329/3300 06:45 < 10:02, 3.27 it/s, Epoch 1.46/4]
Step	Training Loss	Validation Loss	Perplexity
300	5.893800	5.716376	303.802077
600	4.871200	4.750072	115.592654
900	4.480800	4.397294	81.230748
1200	4.243500	4.168933	64.646415


```

**KV Test 1**

* Sharing KV 168,
* sharing FFN ins 192,
* layers --> 8
* decreasing heads/neurons by 25%
* Balanced by decreasing KV

It started well, but eventually fell behind.

```
deepspeed_plugin
 [2361/3300 13:38 < 05:25, 2.88 it/s, Epoch 2.60/4]
Step	Training Loss	Validation Loss	Perplexity
300	5.865600	5.687145	295.050114
600	4.869900	4.751711	115.782164
900	4.477700	4.391935	80.796639
1200	4.229200	4.150353	63.456424
1500	4.058400	3.987595	53.925065
1800	3.959800	3.882795	48.559766
2100	3.865200	3.811624	45.223806
```

**Ninth Test**

Only 8 layers instead of 10 (or 6)
slightly higher intermediate, 9 heads (up from 8)
Query space up to 112.

**Eighth Test**

256, 512, 10 layers, separate in/gate/out at 120, q at 96.

```
deepspeed_plugin
 [2781/3300 16:38 < 03:06, 2.78 it/s, Epoch 3.07/4]
Step	Training Loss	Validation Loss	Perplexity
300	5.890900	5.715315	303.479823
600	4.898200	4.782177	119.363870
900	4.519200	4.437453	84.559284
1200	4.306100	4.241689	69.525208
1500	4.143200	4.082557	59.296904
1800	4.034300	3.967125	52.832402
2100	3.937800	3.888706	48.847657
2400	3.892000	3.831473	46.130454
2700	3.845000	3.790058	44.258974



```

**Seventh Test**

Not bad, but still not any better!

```
10 layers
num_heads = 6
d_model = 256
d_intermediate = 512
q space = 128  (up from 96)

```

**Sixth Test**

Different direction, even more layers!

```
num_layers = 14
num_heads = 4
d_intermediate =

deepspeed_plugin
 [2204/3300 13:44 < 06:50, 2.67 it/s, Epoch 2.43/4]
Step	Training Loss	Validation Loss	Perplexity
300	5.932600	5.769661	320.429200
600	5.001300	4.894136	133.504603
900	4.622900	4.541313	93.813893
1200	4.409600	4.342558	76.903990
1500	4.263100	4.203744	66.936498
1800	4.168400	4.102822	60.510799
2100	4.065600	4.019685	55.683590



```

**Fifth Test**

Slight adjustment to four--added to d_model and decreased d_intermediate.

Performed a little worse.

```
10 layers (vs. 6)
d_model = 260
d_intermediate = 496 (vs. 672)
num_heads = 6 (vs. 8)




```

**Fourth Test**

More subspaces and more layers.
So close to out-performing baseline!

```
10 layers (vs. 6)
d_model = 256
d_intermediate = 544 (vs. 672)
num_heads = 6 (vs. 8)

deepspeed_plugin
 [2487/3300 16:04 < 05:15, 2.58 it/s, Epoch 2.74/4]
Step	Training Loss	Validation Loss	Perplexity
300	5.877400	5.700685	299.072293
600	4.902000	4.786603	119.893454
900	4.513300	4.432715	84.159602
1200	4.282100	4.212252	67.508408
1500	4.106900	4.047623	57.261198
1800	4.006100	3.936759	51.252227
2100	3.910500	3.863502	47.631876
2400	3.866500	3.808168	45.067795

```

**Third Test**

This was a goof--I balanced using 10 heads (thought I was reducing from 12).

```
deepspeed_plugin
 [1801/3300 09:41 < 08:04, 3.09 it/s, Epoch 1.99/4]
Step	Training Loss	Validation Loss	Perplexity
300	6.013900	5.819359	336.756189
600	4.946800	4.830425	125.264138
900	4.519400	4.432698	84.158165
1200	4.231400	4.150273	63.451327
1500	4.053700	3.988451	53.971219

 [4/8 00:02 < 00:03, 1.21 it/s]
```

**2nd Test**

```
deepspeed_plugin
 [3300/3300 19:20, Epoch 3/4]
Step	Training Loss	Validation Loss	Perplexity
300	5.993800	5.801105	330.664730
600	4.915800	4.800222	121.537431
900	4.483200	4.391602	80.769707
1200	4.198300	4.117934	61.432193
1500	4.038600	3.964591	52.698714
1800	3.943300	3.865791	47.741004
2100	3.848400	3.797409	44.585513
2400	3.806100	3.745165	42.316006
2700	3.763300	3.707921	40.768946
3000	3.721500	3.685086	39.848536
3300	3.718400	3.674263	39.419608
 [8/8 00:04]
wandb: WARNING URL not available in offline run


Run history:

epoch	▁
eval/loss	█▅▃▂▂▂▁▁▁▁▁▁
eval/perplexity	█▃▂▂▁▁▁▁▁▁▁▁
eval/runtime	█▄▃▃▅▂▁▁▄▂▁▃
eval/samples_per_second	▁▅▆▆▄▆▇█▅▇█▆
eval/steps_per_second	▁▅▆▆▄▆▇█▅▇█▆
eval_loss	▁
eval_perplexity	▁
eval_runtime	▁
eval_samples_per_second	▁
+6	...

Run summary:

epoch	3.63872
eval/loss	3.67426
eval/perplexity	39.41961
eval/runtime	5.6772
eval/samples_per_second	42.451
eval/steps_per_second	1.409
eval_loss	3.67426
eval_perplexity	39.41961
eval_runtime	5.6772
eval_samples_per_second	42.451
+11	...

You can sync this run to the cloud by running:
wandb sync /content/wandb/offline-run-20251012_055510-zhp5jnml
Find logs at: ./wandb/offline-run-20251012_055510-zhp5jnml/logs
```

**Baseline**

```

deepspeed_plugin
 [3300/3300 17:01, Epoch 3/4]
Step	Training Loss	Validation Loss	Perplexity
300	5.885900	5.696079	297.697706
600	4.862200	4.736460	114.029848
900	4.425000	4.328905	75.861147
1200	4.157900	4.077585	59.002784
1500	4.011000	3.938026	51.317215
1800	3.923400	3.845245	46.770131
2100	3.833700	3.782127	43.909320
2400	3.796200	3.734109	41.850700
2700	3.756200	3.699565	40.429706
3000	3.716400	3.678386	39.582448
3300	3.713500	3.668406	39.189407
 [8/8 00:05]
wandb: WARNING URL not available in offline run


Run history:

epoch	▁
eval/loss	█▅▃▂▂▂▁▁▁▁▁▁
eval/perplexity	█▃▂▂▁▁▁▁▁▁▁▁
eval/runtime	█▁▁▁▂▂▂▂▃▂▆▁
eval/samples_per_second	▁▇██▇▆▇▆▆▆▃▇
eval/steps_per_second	▁███▇▆▇▆▆▆▃▇
eval_loss	▁
eval_perplexity	▁
eval_runtime	▁
eval_samples_per_second	▁
+6	...

Run summary:

epoch	3.63872
eval/loss	3.66841
eval/perplexity	39.18941
eval/runtime	5.6442
eval/samples_per_second	42.698
eval/steps_per_second	1.417
eval_loss	3.66841
eval_perplexity	39.18941
eval_runtime	5.6442
eval_samples_per_second	42.698
+11	...

```